In [2]:
# import pandas and set display options
# define the header to see all columns in the dataframe

import pandas as pd
pd.set_option("display.max_columns", None)

pfad = "data/en.openfoodfacts.org.products.csv.gz"   # an deinen Dateinamen anpassen

header = pd.read_csv(pfad, sep="\t", nrows=0)  # nur die Spaltennamen einlesen


In [3]:
# display the column names of the dataframe
# evaluate the relevance of the columns for the analysis and select the relevant ones


for spalte in header.columns:
    print(spalte)

code
url
creator
created_t
created_datetime
last_modified_t
last_modified_datetime
last_modified_by
last_updated_t
last_updated_datetime
product_name
abbreviated_product_name
generic_name
quantity
packaging
packaging_tags
packaging_en
packaging_text
brands
brands_tags
brands_en
categories
categories_tags
categories_en
origins
origins_tags
origins_en
manufacturing_places
manufacturing_places_tags
labels
labels_tags
labels_en
emb_codes
emb_codes_tags
first_packaging_code_geo
cities
cities_tags
purchase_places
stores
countries
countries_tags
countries_en
ingredients_text
ingredients_tags
ingredients_analysis_tags
allergens
allergens_en
traces
traces_tags
traces_en
serving_size
serving_quantity
no_nutrition_data
additives_n
additives
additives_tags
additives_en
nutriscore_score
nutriscore_grade
nova_group
pnns_groups_1
pnns_groups_2
food_groups
food_groups_tags
food_groups_en
states
states_tags
states_en
brand_owner
environmental_score_score
environmental_score_grade
nutrient_levels_tags
p

In [4]:
#define the relevant columns for the analysis


relevante_spalten = [
    # Identifikation & Kategorien
    "code",
    "product_name",
    "brands",
    "countries_tags",
    "main_category_en",
    "pnns_groups_1",
    "pnns_groups_2",
    "ingredients_analysis_tags",
    "labels_tags",
    # Scores & Verarbeitung
    "nutriscore_score",
    "nutriscore_grade",
    "nova_group",
    "additives_n",
    "environmental_score_score",
    "environmental_score_grade",
    # Hauptnährwerte (EU-Pflichtangaben)
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "fruits-vegetables-legumes_100g",
    # Datenqualität & Popularität
    "completeness",
    "unique_scans_n",
    "data_quality_errors_tags",
    "last_modified_datetime",
]

In [5]:
#read the CSV file in chunks to avoid memory issues


chunks = []
chunk_groesse = 100_000

for i, chunk in enumerate(
    pd.read_csv(
        pfad,
        sep="\t",
        usecols=relevante_spalten,
        chunksize=chunk_groesse,
        on_bad_lines="skip",
        low_memory=True,
    )
):
    # OPTIONAL: Hier direkt filtern, z. B. nur Produkte aus Deutschland (spart 90 % RAM!):
    chunk = chunk[chunk["countries_tags"].str.contains("en:germany", na=False)]

    chunks.append(chunk)
    if (i + 1) % 5 == 0:
        print(f"{(i + 1) * chunk_groesse:,} Zeilen verarbeitet...")

df = pd.concat(chunks, ignore_index=True)
print(f"Fertig! Geladene Zeilen: {len(df):,}")

/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_20741/2280433666.py:7: DtypeWarning: Columns (0: code) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


500,000 Zeilen verarbeitet...
1,000,000 Zeilen verarbeitet...
1,500,000 Zeilen verarbeitet...
2,000,000 Zeilen verarbeitet...
2,500,000 Zeilen verarbeitet...
3,000,000 Zeilen verarbeitet...
3,500,000 Zeilen verarbeitet...
4,000,000 Zeilen verarbeitet...


/var/folders/pt/fb5vwb8s7w5fhj11f4r8_nzw0000gn/T/ipykernel_20741/2280433666.py:7: DtypeWarning: Columns (0: code, 44: ingredients_analysis_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(


4,500,000 Zeilen verarbeitet...
5,000,000 Zeilen verarbeitet...
Fertig! Geladene Zeilen: 439,705


In [6]:
#display the first 10 rows of the dataframe


df.head(10)

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g
0,204286484,2026-03-26T02:25:06Z,Mehrkomponeneten Protein 90 C6 Haselnuß,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.000000
1,250632969,2026-03-26T02:25:07Z,Mehrkomponeneten Protein 90 C6 Banane,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.000000
2,460938714,2026-03-26T02:25:08Z,100% Soja Protein Haselnuss,allfitnessfactory.de,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.8625,Protein powders,362.9,0.5,NaN,1.1,NaN,NaN,88.50,NaN,0.000000
3,470322800,2026-03-26T02:25:09Z,Whey Protein aus Molke Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
4,501050603,2026-03-26T02:25:10Z,Whey Protein aus Molke 1000 Gramm Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
5,526938306,2026-03-26T02:25:12Z,Whey Protein aus Molke 500 Gramm Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.000000
6,790310013,2026-03-26T02:25:12Z,Sour Fruit Gummies,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:non-vegetarian",9.0,19.0,e,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,325.0,0.0,0.0,75.0,57.5,0.0,5.00,0.12700,11.363636
7,790310020,2026-03-26T02:25:12Z,Jelly Fish,Candy Crush,NaN,en:germany,"en:palm-oil-free,en:vegan-status-unknown,en:ve...",7.0,7.0,c,4.0,Sugary snacks,Sweets,73.0,b,NaN,NaN,0.7875,Candies,140.0,0.0,0.0,34.0,24.0,0.0,0.00,0.07620,0.000000
8,1100000136,2018-08-15T07:17:53Z,helle Soße,NaN,NaN,"en:france,en:germany",NaN,NaN,NaN,unknown,NaN,unknown,unknown,NaN,unknown,NaN,NaN,0.2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1182000086,2020-02-01T11:25:20Z,Salat der Saison Dressing für Feldsalat,NaN,NaN,en:germany,NaN,NaN,NaN,unknown,NaN,Fat and sauces,Dressings and sauces,37.0,d,NaN,NaN,0.3000,en:groceries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
#group the dataframe by the column "ingredients_analysis_tags" and display the first 10 rows of each group

gruppen = df.groupby("ingredients_analysis_tags")
gruppen.head(10)

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g
0,204286484,2026-03-26T02:25:06Z,Mehrkomponeneten Protein 90 C6 Haselnuß,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.0
1,250632969,2026-03-26T02:25:07Z,Mehrkomponeneten Protein 90 C6 Banane,allfitnessfactory.de,NaN,en:germany,"en:may-contain-palm-oil,en:non-vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,361.0,1.0,NaN,2.5,NaN,NaN,85.50,NaN,0.0
2,460938714,2026-03-26T02:25:08Z,100% Soja Protein Haselnuss,allfitnessfactory.de,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.8625,Protein powders,362.9,0.5,NaN,1.1,NaN,NaN,88.50,NaN,0.0
3,470322800,2026-03-26T02:25:09Z,Whey Protein aus Molke Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.0
4,501050603,2026-03-26T02:25:10Z,Whey Protein aus Molke 1000 Gramm Vanilla,allfitnessfactory.de,NaN,en:germany,"en:palm-oil-free,en:non-vegan,en:vegetarian-st...",2.0,NaN,not-applicable,4.0,unknown,unknown,NaN,unknown,NaN,NaN,0.9625,Protein powders,389.0,4.6,NaN,8.8,6.0,0.1,78.05,1.21158,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123957,4012362036500,2026-03-26T06:38:13Z,Cornusso,Jetti,en:non-vegan,en:germany,en:non-vegan,NaN,30.0,e,NaN,unknown,unknown,NaN,unknown,NaN,2.0,0.6750,Milk chocolate sprinkles,531.0,29.2,15.1,57.4,44.1,NaN,7.10,0.30000,NaN
125386,4058172823619,2026-03-26T06:49:10Z,Kinder Früchtemüsli,DmBio,"en:organic,en:vegetarian,en:eu-organic,en:vega...",en:germany,"en:palm-oil-content-unknown,en:maybe-vegan,en:...",0.0,NaN,unknown,NaN,Cereals and potatoes,Breakfast cereals,79.0,a,en:vegan-label-but-non-vegan-ingredient,5.0,0.6750,de:pflanzliche-lebensmittel-und-getranke,349.0,5.0,0.9,61.0,13.0,NaN,10.00,NaN,0.0
132067,4316268650410,2026-08-15T12:18:29Z,Suppengrün,Carat,en:non-vegan,en:germany,en:non-vegan,NaN,NaN,unknown,NaN,unknown,unknown,NaN,unknown,NaN,1.0,0.2625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
356328,4000522108761,2026-05-31T19:21:26Z,Pom-Bär Crizzlies - Original,"funny-frisch, Pom-Bär",en:non-vegan,en:germany,en:non-vegan,NaN,NaN,unknown,NaN,Cereals and potatoes,Potatoes,69.0,b,NaN,3.0,0.4750,Stacked extruded potato crisps,517.0,27.0,NaN,62.0,NaN,NaN,4.00,NaN,NaN


In [8]:
#check on the shape of the dataframe

df.shape

(439705, 28)

In [9]:
#check the number of missing values in the dataframe

df.isna().sum()

code                                   0
last_modified_datetime                 0
product_name                       20766
brands                             86554
labels_tags                       312956
countries_tags                         0
ingredients_analysis_tags         295118
additives_n                       306594
nutriscore_score                  311960
nutriscore_grade                    1750
nova_group                        323006
pnns_groups_1                       1710
pnns_groups_2                       1710
environmental_score_score         340341
environmental_score_grade           1720
data_quality_errors_tags          430888
unique_scans_n                    221785
completeness                          26
main_category_en                  250320
energy-kcal_100g                  190932
fat_100g                          194376
saturated-fat_100g                211665
carbohydrates_100g                193482
sugars_100g                       203522
fiber_100g      

In [10]:
# filter the dataframe to only include products with a completeness of at least 0.7, no known data quality errors, and a non-null product name
df_clean = df[
    (df["completeness"] >= 0.7)
    & (df["data_quality_errors_tags"].isna())  # nur Produkte ohne bekannte Datenfehler
    & (df["product_name"].notna())
].copy()

In [11]:
# control if there are unknown values in the columns "nutriscore_grade" and "pnns_groups_1"
df_clean.isna().sum()

code                                  0
last_modified_datetime                0
product_name                          0
brands                              287
labels_tags                       22165
countries_tags                        0
ingredients_analysis_tags          1857
additives_n                        1980
nutriscore_score                  10074
nutriscore_grade                     38
nova_group                         7197
pnns_groups_1                        37
pnns_groups_2                        37
environmental_score_score         29141
environmental_score_grade            38
data_quality_errors_tags          74657
unique_scans_n                     5479
completeness                          0
main_category_en                    259
energy-kcal_100g                   3318
fat_100g                           3593
saturated-fat_100g                 4670
carbohydrates_100g                 3501
sugars_100g                        3994
fiber_100g                        26632


In [12]:
# control if there are unknown values in the columns "nutriscore_grade" and "pnns_groups_1"
print(df_clean["nutriscore_grade"].value_counts(dropna=False))
print(df_clean["pnns_groups_1"].value_counts(dropna=False))

nutriscore_grade
e                 16513
d                 15960
c                 15553
a                 10318
b                  6239
unknown            5971
not-applicable     4065
NaN                  38
Name: count, dtype: int64
pnns_groups_1
Sugary snacks              11214
Cereals and potatoes       10347
Milk and dairy products     9824
unknown                     9112
Beverages                   7711
Fish Meat Eggs              7163
Fruits and vegetables       5102
Fat and sauces              5099
Composite foods             4362
Salty snacks                3071
Alcoholic beverages         1431
Baby foods                   184
NaN                           37
Name: count, dtype: int64


In [13]:
#replace "unknown" and "not-applicable" with NaN in the entire DataFrame

df_clean = df_clean.replace({"unknown": pd.NA, "not-applicable": pd.NA})

In [14]:
#repplace "unknown" and "not-applicable" with NaN in the entire DataFrame
#filter out any unplausible values in the nutritional columns based on physical plausibility limits per 100 g

import numpy as np

# 1. Platzhalter-Strings in echte NaNs umwandeln
df_clean = df_clean.replace({
    "unknown": np.nan,
    "not-applicable": np.nan
})

# 2. Physikalische Plausibilitätsgrenzen pro 100 g prüfen & Ausreißer entfernen
# (Max. 100 g pro 100 g für Makronährstoffe, max. 900 kcal für reines Fett, max. 100 g Salz)
plausibel = (
    (
        df_clean["energy-kcal_100g"].isna()
        | df_clean["energy-kcal_100g"].between(0, 900)
    )
    & (df_clean["fat_100g"].isna() | df_clean["fat_100g"].between(0, 100))
    & (
        df_clean["saturated-fat_100g"].isna()
        | df_clean["saturated-fat_100g"].between(0, 100)
    )
    & (
        df_clean["carbohydrates_100g"].isna()
        | df_clean["carbohydrates_100g"].between(0, 100)
    )
    & (df_clean["sugars_100g"].isna() | df_clean["sugars_100g"].between(0, 100))
    & (df_clean["fiber_100g"].isna() | df_clean["fiber_100g"].between(0, 100))
    & (
        df_clean["proteins_100g"].isna()
        | df_clean["proteins_100g"].between(0, 100)
    )
    & (df_clean["salt_100g"].isna() | df_clean["salt_100g"].between(0, 100))
    & (
        df_clean["fruits-vegetables-legumes_100g"].isna()
        | df_clean["fruits-vegetables-legumes_100g"].between(0, 100)
    )
)

df_clean = df_clean[plausibel].copy()
print(f"Verbleibende plausible Produkte: {len(df_clean):,}")

Verbleibende plausible Produkte: 73,897


In [15]:
#define a function to determine the dietary form based on the ingredients_analysis_tags column


def erkenne_ernaehrungsform(tags):
    if pd.isna(tags):
        return "Unbekannt"

    # Falls tags ein String ist, in eine saubere Menge (Set) aus Einzel-Tags zerlegen
    if isinstance(tags, str):
        tag_set = set(tags.split(","))
    else:
        tag_set = set(tags)

    if "en:vegan" in tag_set:
        return "Vegan"
    if "en:vegetarian" in tag_set:
        return "Vegetarisch"
    if "en:non-vegan" in tag_set or "en:non-vegetarian" in tag_set:
        return "Nicht-vegan"
    return "Unbekannt"


df_clean["diet_type"] = df_clean["ingredients_analysis_tags"].apply(
    erkenne_ernaehrungsform
)
print(df_clean["diet_type"].value_counts())

diet_type
Nicht-vegan    25735
Vegan          22363
Unbekannt      17023
Vegetarisch     8776
Name: count, dtype: int64


In [16]:
mask = df_clean["diet_type"].str.contains("Vegan")
df_clean[mask]

,code,last_modified_datetime,product_name,brands,labels_tags,countries_tags,ingredients_analysis_tags,additives_n,nutriscore_score,nutriscore_grade,nova_group,pnns_groups_1,pnns_groups_2,environmental_score_score,environmental_score_grade,data_quality_errors_tags,unique_scans_n,completeness,main_category_en,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,diet_type
2,460938714,2026-03-26T02:25:08Z,100% Soja Protein Haselnuss,allfitnessfactory.de,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,0.8625,Protein powders,362.90,0.5,NaN,1.10,NaN,NaN,88.50,NaN,0.0,Vegan
12,2983542515,2026-03-26T02:25:14Z,100% Soja Protein Neutral,allfitnessfactory.de,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-content-unknown,en:vegan,en:vegeta...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9625,Protein powders,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Vegan
31,11152134079,2026-03-26T02:25:16Z,Dashi Kombu Dried Seaweed,Welpac,NaN,"en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,14.0,d,1.0,NaN,NaN,NaN,NaN,NaN,3.0,0.7875,en:undefined,200.00,0.0,0.00,40.00,10.0,20.00,0.00,6.500,100.0,Vegan
32,11152443683,2026-08-17T10:19:05Z,Jasmine Rice,Dynasty,"en:no-gmos,en:non-gmo-project","en:germany,en:united-states","en:palm-oil-free,en:vegan,en:vegetarian",0.0,2.0,b,1.0,NaN,NaN,NaN,NaN,NaN,1.0,0.7750,en:undefined,355.56,0.0,0.00,80.00,0.0,2.22,6.67,0.000,0.0,Vegan
34,11210000018,2026-08-27T03:18:10Z,16000085 Tabasco Pepper Sauce,"Mc. Ilhenny Co., Tabasco","en:no-gluten,en:halal,en:no-gmos,en:non-gmo-pr...","en:croatia,en:france,en:germany,en:ireland,en:...","en:palm-oil-free,en:vegan,en:vegetarian",0.0,8.0,c,3.0,Fat and sauces,Dressings and sauces,74.0,b,NaN,20.0,0.7875,en:tabasco pepper,16.00,0.7,0.20,1.60,0.0,1.30,1.00,1.800,19.0,Vegan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436435,5014697017086,2026-06-20T15:33:38Z,Deo - Wasserlilie,CD,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",3.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,1.0,0.7750,en:roll-on-deodorants,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,Vegan
436447,8076809581127,2026-07-19T12:23:10Z,Spaghetti Al Bronzo,Barilla,en:green-dot,"en:germany,en:world,en:Frankreich","en:palm-oil-free,en:vegan,en:vegetarian",0.0,0.0,a,1.0,Cereals and potatoes,Cereals,62.0,b,NaN,1.0,0.7625,Durum wheat spaghetti,359.00,2.0,0.40,70.00,3.5,3.00,14.00,0.010,0.0,Vegan
436529,4021457637461,2023-06-02T07:50:18Z,Body Milk Sanft,Lavera,"en:vegetarian,en:vegan",en:germany,"en:may-contain-palm-oil,en:vegan,en:vegetarian",6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.7750,de:Körperpflege,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vegan
437820,4335619416475,2026-11-04T17:08:15Z,Zero Sirup - Pink Grapefruit Geschmack,Crème de la Cream Company B.V.,"en:vegetarian,en:vegan",en:germany,"en:palm-oil-free,en:vegan,en:vegetarian",10.0,NaN,NaN,4.0,Beverages,Artificially sweetened beverages,80.0,a,NaN,NaN,0.7000,Flavoured syrups,1.00,0.0,0.00,0.00,0.0,0.00,0.00,0.020,6.0,Vegan


In [17]:
df_clean.shape

(73897, 29)

In [18]:
# Pflichtspalten für echte Lebensmittel-Analysen definieren
pflicht_spalten = [
    "pnns_groups_1",
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "proteins_100g",
    "salt_100g",
]

df_clean = df_clean.dropna(subset=pflicht_spalten).copy()

# Sicherheits-Check: Alle noch vorhandenen Kategorien anzeigen
print(df_clean["pnns_groups_1"].value_counts())
print(f"\nFinale saubere Produktanzahl: {len(df_clean):,}")

pnns_groups_1
Sugary snacks              10746
Cereals and potatoes       10012
Milk and dairy products     9533
Fish Meat Eggs              6741
Beverages                   6548
Fat and sauces              4828
Composite foods             4237
Fruits and vegetables       4227
Salty snacks                2990
Alcoholic beverages          327
Baby foods                   170
Name: count, dtype: int64

Finale saubere Produktanzahl: 60,359
